***

# **Congestion Data Formatting Script**

***


This file contains a script for formatting Annual Hours of Peak Hour Excessive Delay Per Capita for use of RITIS data. The script is made to be reproducible yearly, as long as the user has a folder with the necessary files and naming schematics. Specifically, the user must manually download all of the PHED files for the given UZA's, and store all of these files into one folder. This folder for SP, is given by `path_congestion`. In addition, all of these files should be named using the following example schematic: Annual Hours PHED Per Capita_3-7pm_**Austin**_**TX**. The only thing the user will be changing in the file name should be the city and the state abbreviation, evidenced by the bolded characters. The PHED files can be made via the [NPMRDS analytics tool](https://npmrds.ritis.org/analytics/my-dashboard/) and selecting the MAP-21 widget. Search for your UZA, select the Annual Hours of Peak Hour Excessive Delay Per Capita box, and then add all of your years. You will be redirected to see a chart for all of the years selected detailing PHED. You can save this data, in the top right of the panel widget. A good tip to know, is you are able to simply edit your already existing widget to swap out the UZA and the name of the file, rather than re-inputting all of the years again. Additionally, to get the Percent of Eligible Miles missing PHED, you need to make each UZA's dashboard to only feature the last year with full data. So for 2024, we use 2023 data. From there, you look at the bottom right of the chart produced and subtract that number from 100%. 
***

## **Congestion_1**

***

In [1]:
import os
import pandas as pd

In [2]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths

# Git
path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')

# AGOL Path for Pete
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Congestion Data')

# Path with all of the PHED files named via the aforementioned schematic
path_congestion = os.path.join(path_sp, 'Data', 'Safe Equitable Resilient Infrastructure', 'Congestion')
path_phed = os.path.join(path_congestion, 'RTIS', 'PHED')
path_lottr = os.path.join(path_congestion, 'RTIS', 'LOTTR')

print(user)
print(path_git)

jchoy
C:\Users\jchoy\Documents\Projects\Regional-Monitoring\Indicator_Gen


<>:11: SyntaxWarning: invalid escape sequence '\R'
<>:11: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_32400\2183128991.py:11: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [3]:
# Importing the data to follow a naming pattern

df_list = []

# Iterate over every file in path
for filename in os.listdir(path_phed):
    if filename.endswith('.csv'):
        
        # Extract city name from the filename
        uza = filename.split('_')[2]
        
        # Load the file into a df
        file_path = os.path.join(path_phed, filename)
        df = pd.read_csv(file_path)
        
        # Adding new col w/ UZA name. This is for joining later
        df['UZA'] = uza

        df_list.append(df)

In [4]:
# Joining now

congestion_1 = pd.concat(df_list, axis = 0, ignore_index=True)
congestion_1['Month'] = pd.to_datetime(congestion_1['Month']) #format = "%Y/%m"
congestion_1['Month'] = congestion_1['Month'].dt.to_period('M')

congestion_1_wide = congestion_1.pivot_table(index='Month', columns='UZA', values='PHED (hours)', aggfunc='mean', dropna=False)
congestion_1_wide = congestion_1_wide.reset_index()

C:\Users\jchoy\AppData\Local\Temp\ipykernel_32400\3343672330.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  congestion_1['Month'] = pd.to_datetime(congestion_1['Month']) #format = "%Y/%m"


In [5]:
# Get yearly averages for each UZA

congestion_1_year = congestion_1.copy()
congestion_1_year['Year'] = congestion_1_year['Month'].dt.year

# Group by 'Year' and 'UZA', then calculate the mean
# Mean calculation is on the basic side, should consider weighting based off of days in month etc. For now, this is sufficient

congestion_1_year = congestion_1_year.groupby(['Year', 'UZA']).mean().reset_index()
congestion_1_year = congestion_1_year[['Year', 'UZA', 'PHED (hours)']]

# Now we need to do this for the wide format

congestion_1_wide_year = congestion_1_year.pivot_table(index='Year', columns='UZA', values='PHED (hours)', aggfunc='mean', dropna=False)
congestion_1_wide_year = congestion_1_wide_year.reset_index()

***

## **Congestion_3**

***

For Congestion_3, we only need data for SACOG counties. To do this, we use the same tool as in Congestion_1, but now we use the MPA for SACOG instead. Select the first three measures, and do the same as we did before. 

In [77]:
# Loading the files

truck_path = os.path.join(path_lottr, 'Truck Travel Time Reliability - Sacramento.csv')
interstate_path = os.path.join(path_lottr, 'Interstate Travel Time Reliability - Sacramento.csv')
nonint_path = os.path.join(path_lottr, 'Non-interstate NHS Travel Time Reliability - Sacramento.csv')

# reading files

truck_travel = pd.read_csv(truck_path)
interstate_travel = pd.read_csv(interstate_path)
nonint_travel = pd.read_csv(nonint_path)

# specifying LOTTR for int and non-int

interstate_travel.rename(columns={'LOTTR (%)': 'Interstate LOTTR (%)'}, inplace=True)
nonint_travel.rename(columns={'LOTTR (%)': 'Non-Interstate LOTTR (%)'}, inplace=True)

In [78]:
# Merge

congestion_3 = truck_travel.merge(interstate_travel, on='Month', how='left').merge(nonint_travel, on='Month', how='left')
congestion_3['Month'] = pd.to_datetime(congestion_3['Month']) #format = "%Y/%m"
congestion_3['Month'] = congestion_3['Month'].dt.to_period('M')

# Get the averages per year now

congestion_3_year = congestion_3.copy()
congestion_3_year['Year'] = congestion_3_year['Month'].dt.year

# Group by 'Year' and 'UZA', then calculate the mean

congestion_3_year = congestion_3_year.groupby('Year')[['TTTR (%)', 'Interstate LOTTR (%)', 'Non-Interstate LOTTR (%)']].mean().reset_index()
#congestion_3_year = congestion_3_year.drop(columns = 'Month')

C:\Users\jchoy\AppData\Local\Temp\ipykernel_34484\2316967675.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  congestion_3['Month'] = pd.to_datetime(congestion_3['Month']) #format = "%Y/%m"


***

## **Exports**

***

In [86]:
# two exports for data folder | one in date time other in weighted average per year

# one in task 8 for average per year

# we do these exports for both congestion_1 and congestion_3

# Exports
indicator_name = 'Congestion'

# congestion_1
one_output_xlsx = [indicator_name, '_', '1', '.xlsx']
one_output_xlsx = "".join(one_output_xlsx)
one_output_csv = [indicator_name, '_', '1', '.csv']
one_output_csv = "".join(one_output_csv)

# congestion_3
three_output_xlsx = [indicator_name, '_', '3', '.xlsx']
three_output_xlsx = "".join(three_output_xlsx)
three_output_csv = [indicator_name, '_', '3', '.csv']
three_output_csv = "".join(three_output_csv)

In [89]:
# Set file path for exporting

path_out_one = os.path.join(path_congestion, 'Congestion_1')
path_out_three = os.path.join(path_congestion, 'Congestion_3')

# Congestion_1 Exports to SP
with pd.ExcelWriter(os.path.join(path_out_one, one_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_1.to_excel(writer, index = False, sheet_name = 'UZA Long'       )
    congestion_1_wide.to_excel(writer, index = False, sheet_name = 'UZA Wide'  )

# Congestion_1 Exports AGOL
congestion_1_year.to_csv(os.path.join(path_agol, one_output_csv), index = False)
congestion_1_wide_year.to_csv(os.path.join(path_agol, 'Congestion_1 Wide.csv'))

# Congestion_3 Exports
with pd.ExcelWriter(os.path.join(path_out_three, three_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_3.to_excel(writer, index = False, sheet_name = 'SACOG'       )

congestion_3_year.to_csv(os.path.join(path_agol, three_output_csv), index = False)

print('Congestion_1 SP Files Exported Here: ' + path_out_one)
print('Congestion_3 Files Exported Here: '   + path_out_three)
print('Congestion_1 & Congestion_3 AGOL Files Exported Here: ' + path_agol)

Congestion_1 SP Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Safe Equitable Resilient Infrastructure\Congestion\Congestion_1
Congestion_3 Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Safe Equitable Resilient Infrastructure\Congestion\Congestion_3
Congestion_1 & Congestion_3 AGOL Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Process Revamp\Task 8. Reproduce Progress Report indicators\Indicator Data\Congestion Data
